# 6. Feature classification: KinCore classifier experiments <a id="6"></a>


## Table of contents

- [6.1 Train and evaluate Random Forest](#61)
- [6.2 Top-feature distributions and W/KL separation](#7)
- [6.2.1 Setup: paths, KinCore labels, side-chain matrix](#6-2-1-setup-paths-kincore-labels-side-chain-matrix)
- [6.2.2 Side-chain: split violins by PCA cluster](#6-2-2-side-chain-split-violins-by-pca-cluster)
- [6.2.3 Side-chain: split violins by KinCore activation](#6-2-3-side-chain-split-violins-by-kincore-activation)
- [6.2.4 Side-chain: Wasserstein & KL (RF-ranked)](#6-2-4-side-chain-wasserstein-kl-rf-ranked)
- [6.2.5 Cα matrix: train RF and reuse the same split seed](#6-2-5-c-matrix-train-rf-and-reuse-the-same-split-seed)
- [6.2.6 Cα: split violins (PCA cluster + KinCore)](#6-2-6-c-split-violins-pca-cluster-kincore)
- [6.2.7 Cα: Wasserstein & KL (RF-ranked)](#6-2-7-c-wasserstein-kl-rf-ranked)
- [6.3 KinCore MajoritySubsampling](#8)
- [6.3.0 Setup: paths](#6-3-0-setup-paths)
- [6.3.1 Organize Active / Inactive](#6-3-1-organize-active-inactive)
- [6.3.2 Build SC and Cα feature matrices](#6-3-2-build-sc-and-c-feature-matrices)
- [6.3.3 Filter and ANOVA export](#6-3-3-filter-and-anova-export)
- [6.3.4 KinCore RF baseline (side-chain)](#6-3-4-kincore-rf-baseline-side-chain)
- [6.3.5 Undersample and bootstrap compare](#6-3-5-undersample-and-bootstrap-compare)


## Backend map

How this notebook connects to `workflow/` modules (arrows point into the notebook; includes transitive `workflow` subdependencies):

![Backend map](images/backend_maps/11d-KinCoreClassifierExperiments.v2.svg)

<!-- mermaid source (GitHub does not render mermaid in .ipynb; SVG above is for GitHub):
```mermaid
%%{init: {"flowchart": {"nodeSpacing": 12, "rankSpacing": 28, "padding": 4}, "themeVariables": {"fontSize": "11px"}} }%%
flowchart LR
  NB["11d-KinCoreClassifierExperiments.ipynb"]
  m_DunbrackAssignment["DunbrackAssignment"]
  m_analyse_alignment_foldmason["analyse_alignment_foldmason"]
  m_chain_basenames["chain_basenames"]
  m_feature_classification["feature_classification"]
  m_feature_selection["feature_selection"]
  m_guide_tree_clusters["guide_tree_clusters"]
  m_pca_analysis["pca_analysis"]
  m_utilities["utilities"]
  m_chain_basenames --> m_utilities
  m_feature_selection --> m_feature_classification
  m_guide_tree_clusters --> m_feature_classification
  m_utilities --> m_DunbrackAssignment
  m_utilities --> m_pca_analysis
  m_DunbrackAssignment --> NB
  m_analyse_alignment_foldmason --> NB
  m_feature_classification --> NB
  m_feature_selection --> NB
  m_pca_analysis --> NB
  m_utilities --> NB
```
-->


In this section we will train and analyse Random Forest (RF) classifier to investigate what features are most significant in predicting predominant activation loop conformational changes.

The class `FeatureClassification` facilitates running all the steps required to train a RF classifier.

## 6.1 Train and evaluate Random Forest <a id="61"></a>


In [ ]:
import os
import pickle
import shutil
from pathlib import Path
import numpy as np
import pandas as pd

from workflow.DunbrackAssignment import DunbrackWorkflow
from workflow.feature_classification import FeatureClassification
from workflow.feature_selection import FeatureSelection
from workflow.pca_analysis import ClusterAnalyzer
from workflow.utilities import clear_and_make, make_seg, braf_res

classifier = FeatureClassification(
    feature_matrix=fs.feature_matrix,
    labels=fs.labels,
    unique_pairs=fs.unique_pairs,
    fully_conserved=fs.fully_conserved,
    structure_names=fs.structure_names
)

print(f"✅ FeatureClassification initialized")
print(f"   Features: {len(classifier.unique_pairs)}")
print(f"   Structures: {len(classifier.labels)}")
print(f"   Classes: {np.unique(classifier.labels)}")
print(f"   Class distribution: {dict(zip(*np.unique(classifier.labels, return_counts=True)))}")


Let's first split the data into training and validation.

In [ ]:
# Step 1: Split data into train/test sets
classifier.split_data(train_size=0.9, random_state=42)

We can now train the model with our input features.

In [ ]:
# Step 2: Train Random Forest model
classifier.train_model(n_estimators=100, random_state=42)

Let's evaluate model performance and visualise it with a confusion matrix to make sure our classifier is able to deal with the input.

In [ ]:
# Step 3: Evaluate model performance
metrics = classifier.evaluate_model()

# Step 4: Plot confusion matrix
cm = classifier.plot_confusion_matrix()

Let's now visualise and investigate what are the most significant features both looking at Mean Decrease in Impurity (MDI) and SHAP values.

In [ ]:
# Step 5: Compute feature importances (MDI)
importances, importances_std, importances_sem = classifier.compute_feature_importances()

# Step 6: Print top features
top_indices = classifier.print_top_features(n_top=20)

# Step 7: Plot feature ranking
classifier.plot_feature_ranking(n_top=20)

# Step 8: Compute permutation importances
perm_result = classifier.compute_permutation_importances(n_repeats=10, n_jobs=4)

# Step 9: Compute SHAP values (can be slow)
shap_values = classifier.compute_shap_values()
classifier.plot_shap_summary(class_idx=0, max_display=20)  # Class 0
classifier.plot_shap_summary(class_idx=1, max_display=20)  # Class 1
classifier.plot_feature_distributions(n_top=20, class_idx=1)

print("\n" + "="*60)
print("✅ CLASSIFICATION ANALYSIS COMPLETE")
print("="*60)

## 6.2 Top-feature distributions and W/KL separation  <a id="7"></a>

Quantify how well filtered distance features separate structures by:

1. **PCA hierarchical cluster** (cluster 0 vs 1)
2. **KinCore activation state** (inactive vs active)

For each comparison we:

- plot **split histogram violins** (train left / validation right) for the top features by **RF MDI** and **permutation** importance
- compute **Wasserstein** distance and symmetric histogram **KL**, ordered by the same RF rankings

Uses the **random** `train_test_split` from §6 (`random_state=42`, `train_size=0.9`) — not a Newick guide-tree split.

Runs on **side-chain** distances first, then the **Cα** matrix.


## 6.2.1 Setup: paths, KinCore labels, side-chain matrix <a id="6-2-1-setup-paths-kincore-labels-side-chain-matrix"></a>

Load PCA cluster labels and KinCore active/inactive labels aligned to the structures used by the trained RF classifier.


In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

from workflow.feature_classification import FeatureClassification
from workflow.pca_analysis import ClusterAnalyzer

# ── Configuration ─────────────────────────────────────────────────────────
N_TOP_VIOLINS = 20
N_BINS_KL = 31
N_WKL_FEATURES = None  # None => all features, ordered by RF importance

PCA_CLUSTER_LABELS = "cluster_labels_my_analysis_hierarchical.txt"
if not os.path.isfile(PCA_CLUSTER_LABELS):
    PCA_CLUSTER_LABELS = "Results/activation_segments/cluster_labels_my_analysis_hierarchical.txt"

KINCORE_CSV = "Results/dunbrack_assignments/kinase_conformation_assignments.csv"
WKL_ROOT = Path("Results/wkl_analysis")
WKL_SC_DIR = WKL_ROOT / "sidechain"
WKL_CA_DIR = WKL_ROOT / "ca"
WKL_SC_DIR.mkdir(parents=True, exist_ok=True)
WKL_CA_DIR.mkdir(parents=True, exist_ok=True)

# Side-chain matrix from the trained classifier (preferred) or filtered CSV
structure_names = list(classifier.structure_names) if classifier.structure_names else None
if structure_names is None or len(structure_names) != len(classifier.labels):
    if os.path.isfile("filtered_feature_matrix.csv"):
        _tmp = pd.read_csv("filtered_feature_matrix.csv", index_col=0)
        structure_names = list(_tmp.index)
    elif os.path.isfile("corr_filtered_feature_matrix.csv"):
        _tmp = pd.read_csv("corr_filtered_feature_matrix.csv", index_col=0)
        structure_names = list(_tmp.index)
    else:
        structure_names = [f"struct_{i}" for i in range(len(classifier.labels))]

X_df_sc = pd.DataFrame(
    classifier.feature_matrix,
    index=structure_names,
    columns=classifier.feature_names,
)
Xk_sc = X_df_sc.values.astype(float)
feature_labels_sc = [str(c) for c in X_df_sc.columns]
y_sc = np.asarray(classifier.labels)

if classifier.feature_importances is None:
    raise RuntimeError("Run compute_feature_importances() before §7")
if classifier.permutation_result is None:
    raise RuntimeError("Run compute_permutation_importances() before §7")
if classifier.split_labels is None:
    raise RuntimeError("Re-run split_data() so train_idx / split_labels are stored")

gini_mean_sc = np.asarray(classifier.feature_importances, dtype=float)
perm_mean_sc = np.asarray(classifier.permutation_result.importances_mean, dtype=float)
split_labels_sc = np.asarray(classifier.split_labels)
train_idx_sc = np.asarray(classifier.train_idx)
test_idx_sc = np.asarray(classifier.test_idx)
best_k_sc = Xk_sc.shape[1]

# KinCore active/inactive → CSV consumed by distribution helpers
cluster_analyzer = ClusterAnalyzer(n_clusters=2)
bio_labels_sc, _ = cluster_analyzer.load_kincore_labels(
    list(X_df_sc.index), kincore_file=KINCORE_CSV
)
bio_labels_sc = np.asarray(bio_labels_sc, dtype=float)
KINCORE_BIO_CSV = str(WKL_ROOT / "kincore_bio_labels.csv")
pd.DataFrame({"structure": X_df_sc.index, "label": bio_labels_sc.astype(int)}).to_csv(
    KINCORE_BIO_CSV, index=False
)

print(f"Side-chain matrix: {X_df_sc.shape[0]} structures × {X_df_sc.shape[1]} features")
print(f"PCA labels file: {PCA_CLUSTER_LABELS}")
print(f"KinCore CSV: {KINCORE_CSV}")
print(f"KinCore active/inactive: {int((bio_labels_sc == 1).sum())} active, "
      f"{int((bio_labels_sc == 0).sum())} inactive")
print(f"Train/val sizes: {len(train_idx_sc)} / {len(test_idx_sc)}")
print(f"W/KL output: {WKL_SC_DIR}")

## 6.2.2 Side-chain: split violins by PCA cluster <a id="6-2-2-side-chain-split-violins-by-pca-cluster"></a>

Top features by MDI and permutation importance; x-axis = PCA cluster.


In [ ]:
dist_sc_cluster = FeatureClassification.plot_top_feature_distributions_by_label_and_cluster(
    X_df=X_df_sc,
    Xk=Xk_sc,
    feature_labels_all=feature_labels_sc,
    gini_mean=gini_mean_sc,
    perm_mean=perm_mean_sc,
    best_k=best_k_sc,
    biological_labels_csv=KINCORE_BIO_CSV,
    pca_cluster_labels_file=PCA_CLUSTER_LABELS,
    split_labels=split_labels_sc,
    train_idx=train_idx_sc,
    test_idx=test_idx_sc,
    n_top=N_TOP_VIOLINS,
    title_suffix=f"(side-chain RF; k={best_k_sc})",
)
print("✅ Side-chain PCA-cluster distribution plots complete")

## 6.2.3 Side-chain: split violins by KinCore activation <a id="6-2-3-side-chain-split-violins-by-kincore-activation"></a>

Same top RF features; x-axis = inactive vs active.


In [ ]:
dist_sc_activation = FeatureClassification.plot_top_feature_distributions_by_activation(
    X_df=X_df_sc,
    Xk=Xk_sc,
    feature_labels_all=feature_labels_sc,
    gini_mean=gini_mean_sc,
    perm_mean=perm_mean_sc,
    best_k=best_k_sc,
    biological_labels_csv=KINCORE_BIO_CSV,
    split_labels=split_labels_sc,
    train_idx=train_idx_sc,
    test_idx=test_idx_sc,
    n_top=N_TOP_VIOLINS,
    title_suffix=f"(side-chain RF; k={best_k_sc})",
)
# Keep cluster_binary from PCA distribution results for W/KL
dist_sc_activation["cluster_binary"] = dist_sc_cluster["cluster_binary"]
print("✅ Side-chain KinCore activation distribution plots complete")

## 6.2.4 Side-chain: Wasserstein & KL (RF-ranked) <a id="6-2-4-side-chain-wasserstein-kl-rf-ranked"></a>

Line plots of train/val separation vs RF importance rank for PCA clusters and KinCore labels (MDI and permutation orderings).


In [ ]:
def _run_wkl_plots(
    *,
    X_df,
    dist_cluster,
    dist_activation,
    gini_mean,
    perm_mean,
    best_k,
    out_dir: Path,
    tag: str,
):
    n_pool = X_df.shape[1] if N_WKL_FEATURES is None else min(N_WKL_FEATURES, X_df.shape[1])
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    jobs = [
        ("mdi", gini_mean, "mdi_rank", "MDI (Gini) rank (1 = highest)", "MDI-ranked"),
        ("perm", perm_mean, "perm_rank", "Permutation rank (1 = highest)", "permutation-ranked"),
    ]

    results = {}
    for key, importance, rank_col, rank_xlabel, pool_caption in jobs:
        print(f"\n{tag} — {pool_caption}: PCA cluster 0 vs 1")
        wkl_cluster = FeatureClassification.compute_cluster0_vs_cluster1_wasserstein_kl_for_rf_features(
            X_df=X_df,
            distribution_plot_results=dist_cluster,
            importance=importance,
            rank_col=rank_col,
            n_features=n_pool,
            n_bins_kl=N_BINS_KL,
        )
        csv_c = out_dir / f"wkl_pca_cluster_{key}.csv"
        wkl_cluster.to_csv(csv_c, index=False)
        FeatureClassification.plot_cluster0_vs_cluster1_wasserstein_kl(
            wkl_cluster,
            best_k=best_k,
            n_pool_features=n_pool,
            rank_col=rank_col,
            rank_xlabel=rank_xlabel,
            pool_caption=pool_caption,
            comparison_label="cluster 0 vs 1",
            split_caption="random train/test split",
            save_path=str(out_dir / f"wkl_pca_cluster_{key}.png"),
        )

        print(f"\n{tag} — {pool_caption}: KinCore inactive vs active")
        wkl_act = FeatureClassification.compute_active_vs_inactive_wasserstein_kl_for_rf_features(
            X_df=X_df,
            distribution_plot_results=dist_activation,
            importance=importance,
            rank_col=rank_col,
            n_features=n_pool,
            n_bins_kl=N_BINS_KL,
        )
        csv_a = out_dir / f"wkl_kincore_activation_{key}.csv"
        wkl_act.to_csv(csv_a, index=False)
        FeatureClassification.plot_cluster0_vs_cluster1_wasserstein_kl(
            wkl_act,
            best_k=best_k,
            n_pool_features=n_pool,
            rank_col=rank_col,
            rank_xlabel=rank_xlabel,
            pool_caption=pool_caption,
            comparison_label="inactive vs active",
            split_caption="random train/test split",
            save_path=str(out_dir / f"wkl_kincore_activation_{key}.png"),
        )
        results[key] = {"cluster": wkl_cluster, "activation": wkl_act}
        print(f"Saved CSVs: {csv_c.name}, {csv_a.name}")

    return results


wkl_sc = _run_wkl_plots(
    X_df=X_df_sc,
    dist_cluster=dist_sc_cluster,
    dist_activation=dist_sc_activation,
    gini_mean=gini_mean_sc,
    perm_mean=perm_mean_sc,
    best_k=best_k_sc,
    out_dir=WKL_SC_DIR,
    tag="Side-chain",
)
print("\n✅ Side-chain Wasserstein / KL complete")

## 6.2.5 Cα matrix: train RF and reuse the same split seed <a id="6-2-5-c-matrix-train-rf-and-reuse-the-same-split-seed"></a>

Load the Cα feature matrix (prefer filtered/corr-filtered if present), train a second RF with the same `train_size` / `random_state`, then repeat violins + W/KL.


In [ ]:
# Prefer filtered Cα if present; otherwise raw ca_feature_matrix.csv
_ca_candidates = [
    "ca_corr_filtered_feature_matrix.csv",
    "ca_filtered_feature_matrix.csv",
    "ca_feature_matrix.csv",
]
_ca_path = next((p for p in _ca_candidates if os.path.isfile(p)), None)
if _ca_path is None:
    raise FileNotFoundError(
        "No Cα feature matrix found. Expected one of: " + ", ".join(_ca_candidates)
    )
if _ca_path == "ca_feature_matrix.csv":
    print(f"⚠️  Using raw {_ca_path} (no ca_*filtered* matrix found)")

X_df_ca = pd.read_csv(_ca_path, index_col=0)

# Align to side-chain structures when possible
shared = [s for s in X_df_sc.index if s in X_df_ca.index]
if len(shared) < 10:
    # try normalized keys
    sc_norm = {FeatureClassification._normalize_structure_name(s): s for s in X_df_sc.index}
    ca_norm = {FeatureClassification._normalize_structure_name(s): s for s in X_df_ca.index}
    shared_keys = sorted(set(sc_norm) & set(ca_norm))
    shared = [ca_norm[k] for k in shared_keys]
    X_df_ca = X_df_ca.loc[shared]
    # remap SC-aligned labels via normalized names
    sc_to_label = {
        FeatureClassification._normalize_structure_name(s): y
        for s, y in zip(X_df_sc.index, y_sc)
    }
    y_ca = np.array([sc_to_label[FeatureClassification._normalize_structure_name(s)] for s in shared])
else:
    X_df_ca = X_df_ca.loc[shared]
    y_ca = np.asarray(y_sc[[list(X_df_sc.index).index(s) for s in shared]])

# Parse unique pairs from column names "i-j"
unique_pairs_ca = []
for col in X_df_ca.columns:
    parts = str(col).split("-")
    if len(parts) == 2 and parts[0].isdigit() and parts[1].isdigit():
        unique_pairs_ca.append((int(parts[0]), int(parts[1])))
    else:
        unique_pairs_ca.append((col, col))

fully_conserved_ca = getattr(classifier, "fully_conserved", None) or []

classifier_ca = FeatureClassification(
    feature_matrix=X_df_ca.values.astype(float),
    labels=y_ca,
    unique_pairs=unique_pairs_ca,
    fully_conserved=fully_conserved_ca,
    structure_names=list(X_df_ca.index),
)
classifier_ca.split_data(train_size=0.9, random_state=42)
classifier_ca.train_model(n_estimators=100, random_state=42)
_ = classifier_ca.evaluate_model()
gini_mean_ca, _, _ = classifier_ca.compute_feature_importances()
perm_result_ca = classifier_ca.compute_permutation_importances(n_repeats=10, n_jobs=4)
perm_mean_ca = np.asarray(perm_result_ca.importances_mean, dtype=float)

Xk_ca = X_df_ca.values.astype(float)
feature_labels_ca = [str(c) for c in X_df_ca.columns]
split_labels_ca = np.asarray(classifier_ca.split_labels)
train_idx_ca = np.asarray(classifier_ca.train_idx)
test_idx_ca = np.asarray(classifier_ca.test_idx)
best_k_ca = Xk_ca.shape[1]

# KinCore labels for Cα structures
bio_labels_ca, _ = cluster_analyzer.load_kincore_labels(
    list(X_df_ca.index), kincore_file=KINCORE_CSV
)
bio_labels_ca = np.asarray(bio_labels_ca, dtype=float)
KINCORE_BIO_CSV_CA = str(WKL_CA_DIR / "kincore_bio_labels.csv")
pd.DataFrame({"structure": X_df_ca.index, "label": bio_labels_ca.astype(int)}).to_csv(
    KINCORE_BIO_CSV_CA, index=False
)

print(f"Cα matrix ({_ca_path}): {X_df_ca.shape[0]} × {X_df_ca.shape[1]}")
print(f"Train/val: {len(train_idx_ca)} / {len(test_idx_ca)}")
print(f"W/KL output: {WKL_CA_DIR}")

## 6.2.6 Cα: split violins (PCA cluster + KinCore) <a id="6-2-6-c-split-violins-pca-cluster-kincore"></a>


In [ ]:
dist_ca_cluster = FeatureClassification.plot_top_feature_distributions_by_label_and_cluster(
    X_df=X_df_ca,
    Xk=Xk_ca,
    feature_labels_all=feature_labels_ca,
    gini_mean=gini_mean_ca,
    perm_mean=perm_mean_ca,
    best_k=best_k_ca,
    biological_labels_csv=KINCORE_BIO_CSV_CA,
    pca_cluster_labels_file=PCA_CLUSTER_LABELS,
    split_labels=split_labels_ca,
    train_idx=train_idx_ca,
    test_idx=test_idx_ca,
    n_top=N_TOP_VIOLINS,
    title_suffix=f"(Cα RF; k={best_k_ca})",
)

dist_ca_activation = FeatureClassification.plot_top_feature_distributions_by_activation(
    X_df=X_df_ca,
    Xk=Xk_ca,
    feature_labels_all=feature_labels_ca,
    gini_mean=gini_mean_ca,
    perm_mean=perm_mean_ca,
    best_k=best_k_ca,
    biological_labels_csv=KINCORE_BIO_CSV_CA,
    split_labels=split_labels_ca,
    train_idx=train_idx_ca,
    test_idx=test_idx_ca,
    n_top=N_TOP_VIOLINS,
    title_suffix=f"(Cα RF; k={best_k_ca})",
)
print("✅ Cα distribution plots complete")

## 6.2.7 Cα: Wasserstein & KL (RF-ranked) <a id="6-2-7-c-wasserstein-kl-rf-ranked"></a>


In [ ]:
wkl_ca = _run_wkl_plots(
    X_df=X_df_ca,
    dist_cluster=dist_ca_cluster,
    dist_activation=dist_ca_activation,
    gini_mean=gini_mean_ca,
    perm_mean=perm_mean_ca,
    best_k=best_k_ca,
    out_dir=WKL_CA_DIR,
    tag="Cα",
)
print("\n✅ Cα Wasserstein / KL complete")
print("=" * 60)
print("✅ SECTION 7 COMPLETE (side-chain + Cα distributions and W/KL)")
print("=" * 60)

## 6.3 KinCore MajoritySubsampling  <a id="8"></a>

KinCore-label experiment: organize Active/Inactive structures, build SC + Cα distance features, filter, train a baseline RF, then compare **no subsampling** vs **majority undersampling** vs a **bootstrap majority ensemble** (`N_BOOTSTRAP=30`).

Sections **6–7** above remain the PCA-cluster RF + W/KL analysis. This section is independent and writes under `Results/Experiments/KinCoreClassifier/`.


## 6.3.0 Setup: paths <a id="6-3-0-setup-paths"></a>

Resolve the structure pool, KinCore assignments CSV, optional conservation reference pickle, and FoldMason MSA (load only — do not re-run multi-MSA). Create `KCC_OUTPUT_DIR`.


In [ ]:
import os
import shutil
from glob import glob
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from workflow.utilities import clear_and_make, make_seg, braf_res
from workflow.DunbrackAssignment import DunbrackWorkflow
from workflow.feature_selection import FeatureSelection
from workflow.feature_classification import FeatureClassification


def _first_existing(paths):
    for p in paths:
        if p and os.path.exists(p):
            return p
    return None


def _resolve_structure_pool():
    preferred = [
        "Results/activation_segments/misaligned_filter/",
        "Results/activation_segments/reconstr_MODELLER_aligned/",
    ]
    for p in preferred:
        if os.path.isdir(p) and any(f.endswith(".pdb") for f in os.listdir(p)):
            return p.rstrip("/") + "/"
    root = "Results/activation_segments"
    if os.path.isdir(root):
        for name in sorted(os.listdir(root)):
            cand = os.path.join(root, name)
            if not os.path.isdir(cand):
                continue
            pdbs = [f for f in os.listdir(cand) if f.endswith(".pdb")]
            if pdbs:
                return cand.rstrip("/") + "/"
    return None


def _resolve_msa_3di():
    return _first_existing([
        "Results/activation_segments/multi_aligned_foldmason/msa_3di.fa",
        "Results/multi_aligned_foldmason/msa_3di.fa",
    ])


KCC_OUTPUT_DIR = "Results/Experiments/KinCoreClassifier"
os.makedirs(KCC_OUTPUT_DIR, exist_ok=True)

STRUCTURE_POOL = _resolve_structure_pool()
KINCORE_CSV = _first_existing([
    "Results/dunbrack_assignments/kinase_conformation_assignments.csv",
])
CONS_REF_PKL = _first_existing([
    os.path.join(KCC_OUTPUT_DIR, "kcc_reference_data.pkl"),
    os.path.join(KCC_OUTPUT_DIR, "kcc_corr_filtered_reference_data.pkl"),
    "corr_filtered_reference_data.pkl",
    "ca_reference_data.pkl",
    "reference_data.pkl",
    "filtered_reference_data.pkl",
])
MSA_3DI = _resolve_msa_3di()

N_BOOTSTRAP = 30
RANDOM_STATE = 42

if KINCORE_CSV is None:
    raise FileNotFoundError(
        "KinCore assignments CSV not found. Expected "
        "Results/dunbrack_assignments/kinase_conformation_assignments.csv"
    )
if STRUCTURE_POOL is None:
    raise FileNotFoundError(
        "Structure pool not found. Tried misaligned_filter/, "
        "reconstr_MODELLER_aligned/, then any PDB dir under "
        "Results/activation_segments/"
    )

print("KCC_OUTPUT_DIR :", KCC_OUTPUT_DIR)
print("STRUCTURE_POOL :", STRUCTURE_POOL)
print("KINCORE_CSV    :", KINCORE_CSV)
print("CONS_REF_PKL   :", CONS_REF_PKL)
print("MSA_3DI        :", MSA_3DI)
print("N_BOOTSTRAP    :", N_BOOTSTRAP)


## 6.3.1 Organize Active / Inactive <a id="6-3-1-organize-active-inactive"></a>

Map KinCore activation state onto PDBs from the structure pool and copy into `structuresToFeaturise{Active,Inactive}` under `KCC_OUTPUT_DIR`.


In [ ]:
df_labels = DunbrackWorkflow.load_and_annotate_assignments(KINCORE_CSV)

active_dir = os.path.join(KCC_OUTPUT_DIR, "structuresToFeaturiseActive")
inactive_dir = os.path.join(KCC_OUTPUT_DIR, "structuresToFeaturiseInactive")
clear_and_make(active_dir)
clear_and_make(inactive_dir)

source_dir = STRUCTURE_POOL
pool_files = [f for f in os.listdir(source_dir) if f.endswith(".pdb")]

copied_active, copied_inactive, missing_files = [], [], []

for _, row in df_labels.iterrows():
    activation_state = row["activation_state"]
    pdb_file = str(row["pdb_file"])
    structure_name = pdb_file if pdb_file.endswith(".pdb") else pdb_file + ".pdb"
    pdb_prefix = structure_name[:6]
    matching_files = [f for f in pool_files if f.startswith(pdb_prefix)]
    if not matching_files:
        missing_files.append(structure_name)
        continue
    src = os.path.join(source_dir, matching_files[0])
    if activation_state == "Active":
        shutil.copy2(src, os.path.join(active_dir, matching_files[0]))
        copied_active.append(matching_files[0])
    elif activation_state == "Inactive":
        shutil.copy2(src, os.path.join(inactive_dir, matching_files[0]))
        copied_inactive.append(matching_files[0])

print("=== Activation State Organization ===")
print(f"Source: {source_dir}")
print(f"Active:   {len(copied_active)} → {active_dir}")
print(f"Inactive: {len(copied_inactive)} → {inactive_dir}")
print(f"Total organized: {len(copied_active) + len(copied_inactive)}")
print("\nActivation state counts in assignments CSV:")
print(df_labels["activation_state"].value_counts().to_string())
if missing_files:
    print(f"\n⚠️  {len(missing_files)} files not found in pool (showing first 5):")
    for mf in missing_files[:5]:
        print(f"  - {mf[:6]}* (from {mf})")
if len(copied_active) == 0 or len(copied_inactive) == 0:
    raise RuntimeError("Need both Active and Inactive structures after organization")


## 6.3.2 Build SC and Cα feature matrices <a id="6-3-2-build-sc-and-c-feature-matrices"></a>

Reuse `fully_conserved` from an existing reference pickle when available (skip FoldMason multi-MSA). Load alignment objects from an existing `msa_3di.fa` only as needed for residue mapping. Rebuild distances for the Active/Inactive folders, combine, and save `kcc_*` / `kcc_ca_*`.

If `kcc_feature_matrix.csv` already exists under `KCC_OUTPUT_DIR`, reload instead of recomputing.


In [ ]:
kcc_fm = os.path.join(KCC_OUTPUT_DIR, "kcc_feature_matrix.csv")
kcc_ca_fm = os.path.join(KCC_OUTPUT_DIR, "kcc_ca_feature_matrix.csv")

if os.path.isfile(kcc_fm) and os.path.isfile(kcc_ca_fm):
    print("Found existing kcc_* / kcc_ca_* matrices — reloading")
    fs = FeatureSelection(dfg_index=145, ape_index=174, conservation_threshold=0.97)
    fs.load_results(os.path.join(KCC_OUTPUT_DIR, "kcc_reference_data.pkl"))
    feature_df = pd.read_csv(kcc_fm, index_col=0)
    fs.feature_matrix = feature_df.values
    fs.structure_names = list(feature_df.index)
    labels_df = pd.read_csv(os.path.join(KCC_OUTPUT_DIR, "kcc_labels.csv"))
    fs.labels = labels_df["label"].values

    fs_ca = FeatureSelection(dfg_index=145, ape_index=174, conservation_threshold=0.97)
    fs_ca.load_results(os.path.join(KCC_OUTPUT_DIR, "kcc_ca_reference_data.pkl"))
    ca_feature_df = pd.read_csv(kcc_ca_fm, index_col=0)
    fs_ca.feature_matrix = ca_feature_df.values
    fs_ca.structure_names = list(ca_feature_df.index)
    ca_labels_df = pd.read_csv(os.path.join(KCC_OUTPUT_DIR, "kcc_ca_labels.csv"))
    fs_ca.labels = ca_labels_df["label"].values
    print(f"SC: {fs.feature_matrix.shape}  Cα: {fs_ca.feature_matrix.shape}")
else:
    # --- conservation / alignment ---
    fs = FeatureSelection(dfg_index=145, ape_index=174, conservation_threshold=0.97)
    if CONS_REF_PKL is not None:
        with open(CONS_REF_PKL, "rb") as f:
            _ref = pickle.load(f)
        fully = _ref.get("fully_conserved")
        if not fully:
            raise ValueError(f"No fully_conserved in {CONS_REF_PKL}")
        fs.fully_conserved = fully
        print(f"Loaded fully_conserved ({len(fully)}) from {CONS_REF_PKL}")
    else:
        if MSA_3DI is None:
            raise FileNotFoundError(
                "No conservation pickle and no msa_3di.fa — cannot define conserved residues. "
                "Run August 08a/09 FeatureSelection first, or provide corr_filtered_reference_data.pkl."
            )
        from workflow.analyse_alignment_foldmason import analyse_alignment
        analyser = analyse_alignment()
        conservation_run = analyser.run_multi_alignment_conservation_analysis(
            alignment_file=MSA_3DI,
            reference_name="6UAN_chainD",
            reference_residues=braf_res(),
            conservation_threshold=0.70,
            output_plot=os.path.join(KCC_OUTPUT_DIR, "kcc_conservation.png"),
            output_csv=os.path.join(KCC_OUTPUT_DIR, "kcc_conserved_residues.csv"),
            show_plot=False,
        )
        fs.identify_conserved_residues(
            conservation=conservation_run["conservation"],
            reference_residues=braf_res(),
        )
        print(f"Identified {len(fs.fully_conserved)} conserved residues from MSA")

    if MSA_3DI is None:
        raise FileNotFoundError(
            "msa_3di.fa required to map conserved residues onto PDBs for distance calculation. "
            "Expected Results/activation_segments/multi_aligned_foldmason/msa_3di.fa"
        )
    from workflow.analyse_alignment_foldmason import analyse_alignment
    analyser = analyse_alignment()
    multi_data = analyser.load_multi_alignment(MSA_3DI, reference_name="6UAN_chainD")
    if multi_data is None:
        raise ValueError(f"Could not load multi alignment from {MSA_3DI}")
    aligned_structures = multi_data["structures"]
    print(f"Loaded {len(aligned_structures)} alignment objects from {MSA_3DI}")

    fs_ca = FeatureSelection(dfg_index=145, ape_index=174, conservation_threshold=0.97)
    fs_ca.fully_conserved = fs.fully_conserved

    # Active SC + Cα
    print("\n--- Active distances ---")
    distance_df_active = fs.calculate_intra_structure_distances(
        aligned_structures=aligned_structures,
        pdb_directory=active_dir,
        alignment_function=make_seg,
    )
    ca_distance_df_active = fs.calculate_intra_structure_ca_distances()

    # Inactive SC + Cα
    print("\n--- Inactive distances ---")
    distance_df_inactive = fs.calculate_intra_structure_distances(
        aligned_structures=aligned_structures,
        pdb_directory=inactive_dir,
        alignment_function=make_seg,
    )
    ca_distance_df_inactive = fs.calculate_intra_structure_ca_distances()

    fs.intra_structure_df = pd.concat(
        [distance_df_active, distance_df_inactive], ignore_index=True
    )
    fs_ca.intra_structure_df = pd.concat(
        [ca_distance_df_active, ca_distance_df_inactive], ignore_index=True
    )

    cluster_dirs = {0: active_dir, 1: inactive_dir}
    fs.assign_labels_from_clusters(cluster_dirs)
    fs_ca.assign_labels_from_clusters(cluster_dirs)

    fs.build_feature_matrix(use_median_imputation=True)
    fs_ca.build_feature_matrix(use_median_imputation=True)

    fs.save_results(output_prefix=os.path.join(KCC_OUTPUT_DIR, "kcc_"))
    fs_ca.save_results(output_prefix=os.path.join(KCC_OUTPUT_DIR, "kcc_ca_"))
    print(f"Saved SC {fs.feature_matrix.shape} and Cα {fs_ca.feature_matrix.shape}")


## 6.3.3 Filter and ANOVA export <a id="6-3-3-filter-and-anova-export"></a>

Outlier → correlation → ANOVA (top 300) on SC and Cα; impute; save `kcc_filtered_*` / `kcc_ca_filtered_*`; summarize filtering steps.


In [ ]:
# Reload raw matrices if jumping here
if "fs" not in dir() or fs.feature_matrix is None:
    fs = FeatureSelection(dfg_index=145, ape_index=174, conservation_threshold=0.97)
    fs.load_results(os.path.join(KCC_OUTPUT_DIR, "kcc_reference_data.pkl"))
    feature_df = pd.read_csv(os.path.join(KCC_OUTPUT_DIR, "kcc_feature_matrix.csv"), index_col=0)
    fs.feature_matrix = feature_df.values
    fs.structure_names = list(feature_df.index)
    labels_df = pd.read_csv(os.path.join(KCC_OUTPUT_DIR, "kcc_labels.csv"))
    fs.labels = labels_df["label"].values

if "fs_ca" not in dir() or fs_ca.feature_matrix is None:
    fs_ca = FeatureSelection(dfg_index=145, ape_index=174, conservation_threshold=0.97)
    fs_ca.load_results(os.path.join(KCC_OUTPUT_DIR, "kcc_ca_reference_data.pkl"))
    ca_feature_df = pd.read_csv(os.path.join(KCC_OUTPUT_DIR, "kcc_ca_feature_matrix.csv"), index_col=0)
    fs_ca.feature_matrix = ca_feature_df.values
    fs_ca.structure_names = list(ca_feature_df.index)
    ca_labels_df = pd.read_csv(os.path.join(KCC_OUTPUT_DIR, "kcc_ca_labels.csv"))
    fs_ca.labels = ca_labels_df["label"].values

# Skip full filter if filtered matrices already exist
kcc_filt = os.path.join(KCC_OUTPUT_DIR, "kcc_filtered_feature_matrix.csv")
kcc_ca_filt = os.path.join(KCC_OUTPUT_DIR, "kcc_ca_filtered_feature_matrix.csv")

if os.path.isfile(kcc_filt) and os.path.isfile(kcc_ca_filt):
    print("Found existing filtered matrices — reloading")
    fs = FeatureSelection(dfg_index=145, ape_index=174, conservation_threshold=0.97)
    fs.load_results(os.path.join(KCC_OUTPUT_DIR, "kcc_filtered_reference_data.pkl"))
    feature_df = pd.read_csv(kcc_filt, index_col=0)
    fs.feature_matrix = feature_df.values
    fs.structure_names = list(feature_df.index)
    labels_df = pd.read_csv(os.path.join(KCC_OUTPUT_DIR, "kcc_filtered_labels.csv"))
    fs.labels = labels_df["label"].values

    fs_ca = FeatureSelection(dfg_index=145, ape_index=174, conservation_threshold=0.97)
    fs_ca.load_results(os.path.join(KCC_OUTPUT_DIR, "kcc_ca_filtered_reference_data.pkl"))
    ca_feature_df = pd.read_csv(kcc_ca_filt, index_col=0)
    fs_ca.feature_matrix = ca_feature_df.values
    fs_ca.structure_names = list(ca_feature_df.index)
    ca_labels_df = pd.read_csv(os.path.join(KCC_OUTPUT_DIR, "kcc_ca_filtered_labels.csv"))
    fs_ca.labels = ca_labels_df["label"].values
else:
    fs.filter_outlier_distances_and_drop_nan(
        threshold=50.0, set_to_nan=True, max_nan_fraction=1.0,
        outliers_csv_path=os.path.join(KCC_OUTPUT_DIR, "kcc_outlier_distances.csv"),
        print_top_n=10, verbose=True,
    )
    fs_ca.filter_outlier_distances_and_drop_nan(
        threshold=50.0, set_to_nan=True, max_nan_fraction=1.0,
        outliers_csv_path=os.path.join(KCC_OUTPUT_DIR, "kcc_ca_outlier_distances.csv"),
        print_top_n=10, verbose=True,
    )

    selected_features, _ = fs.filter_correlated_features(
        correlation_threshold=0.90, plot_histogram=False, plot_network=False,
        use_parallel=True, n_jobs=-1,
    )
    fs.apply_feature_selection(selected_features)
    fs.save_results(output_prefix=os.path.join(KCC_OUTPUT_DIR, "kcc_corr_filtered_"))

    ca_selected, _ = fs_ca.filter_correlated_features(
        correlation_threshold=0.90, plot_histogram=False, plot_network=False,
        use_parallel=True, n_jobs=-1,
    )
    fs_ca.apply_feature_selection(ca_selected)
    fs_ca.save_results(output_prefix=os.path.join(KCC_OUTPUT_DIR, "kcc_ca_corr_filtered_"))

    fs.filter_anova_features(n_features=300, plot_scores=False, remove=True)
    fs_ca.filter_anova_features(n_features=300, plot_scores=False, remove=True)
    fs.impute_remaining_nan(strategy="median")
    fs_ca.impute_remaining_nan(strategy="median")

    fs.save_results(output_prefix=os.path.join(KCC_OUTPUT_DIR, "kcc_filtered_"))
    fs_ca.save_results(output_prefix=os.path.join(KCC_OUTPUT_DIR, "kcc_ca_filtered_"))
    print(f"Filtered SC: {fs.feature_matrix.shape}  Cα: {fs_ca.feature_matrix.shape}")

filtering_summary = FeatureSelection.summarize_filtering_and_export_anova_features(
    base_dir=KCC_OUTPUT_DIR,
    initial_matrix_csv="kcc_feature_matrix.csv",
    corr_filtered_matrix_csv="kcc_corr_filtered_feature_matrix.csv",
    anova_filtered_matrix_csv="kcc_filtered_feature_matrix.csv",
    reference_pickle="kcc_filtered_reference_data.pkl",
    summary_plot_png="kcc_feature_counts_by_filtering_step.png",
    anova_features_csv="kcc_anova_selected_features_full_list.csv",
    show_plot=True,
)


## 6.3.4 KinCore RF baseline (side-chain) <a id="6-3-4-kincore-rf-baseline-side-chain"></a>

Train a baseline Random Forest on the KinCore-labeled SC filtered matrix (`random_state=42`, 90/10 split). This establishes the shared held-out test set for §8.5.


In [ ]:
fs_kcc = FeatureSelection(dfg_index=145, ape_index=174, conservation_threshold=0.97)
fs_kcc.load_results(os.path.join(KCC_OUTPUT_DIR, "kcc_filtered_reference_data.pkl"))
feature_df = pd.read_csv(os.path.join(KCC_OUTPUT_DIR, "kcc_filtered_feature_matrix.csv"), index_col=0)
fs_kcc.feature_matrix = feature_df.values
fs_kcc.structure_names = list(feature_df.index)
labels_df = pd.read_csv(os.path.join(KCC_OUTPUT_DIR, "kcc_filtered_labels.csv"))
fs_kcc.labels = labels_df["label"].values

baseline_sc = FeatureClassification(
    feature_matrix=fs_kcc.feature_matrix,
    labels=fs_kcc.labels,
    unique_pairs=fs_kcc.unique_pairs,
    fully_conserved=fs_kcc.fully_conserved,
    structure_names=fs_kcc.structure_names,
)
baseline_sc.split_data(train_size=0.9, random_state=RANDOM_STATE)
baseline_sc.train_model(n_estimators=100, random_state=RANDOM_STATE)
metrics_sc_baseline = baseline_sc.evaluate_model()
cm_sc_baseline = baseline_sc.plot_confusion_matrix()
print("Train class distribution:",
      dict(zip(*np.unique(baseline_sc.train_class, return_counts=True))))


## 6.3.5 Undersample and bootstrap compare <a id="6-3-5-undersample-and-bootstrap-compare"></a>

Compare baseline / majority undersampling / bootstrap ensemble on the **shared** test set (subsampling applied to train only). Repeat for the Cα filtered matrix. Outputs under `KCC_OUTPUT_DIR/majority_subsampling/{sidechain,ca}/`.


In [ ]:
save_sc = os.path.join(KCC_OUTPUT_DIR, "majority_subsampling", "sidechain")
cmp_sc = FeatureClassification.compare_majority_subsampling_models(
    baseline_sc,
    n_bootstrap=N_BOOTSTRAP,
    n_estimators=100,
    random_state=RANDOM_STATE,
    save_dir=save_sc,
    show_plot=True,
)
print("\nSide-chain comparison complete →", save_sc)


In [ ]:
# Cα baseline + comparison (same split seed; independent FeatureClassification)
fs_kcc_ca = FeatureSelection(dfg_index=145, ape_index=174, conservation_threshold=0.97)
fs_kcc_ca.load_results(os.path.join(KCC_OUTPUT_DIR, "kcc_ca_filtered_reference_data.pkl"))
ca_feature_df = pd.read_csv(
    os.path.join(KCC_OUTPUT_DIR, "kcc_ca_filtered_feature_matrix.csv"), index_col=0
)
fs_kcc_ca.feature_matrix = ca_feature_df.values
fs_kcc_ca.structure_names = list(ca_feature_df.index)
ca_labels_df = pd.read_csv(os.path.join(KCC_OUTPUT_DIR, "kcc_ca_filtered_labels.csv"))
fs_kcc_ca.labels = ca_labels_df["label"].values

baseline_ca = FeatureClassification(
    feature_matrix=fs_kcc_ca.feature_matrix,
    labels=fs_kcc_ca.labels,
    unique_pairs=fs_kcc_ca.unique_pairs,
    fully_conserved=fs_kcc_ca.fully_conserved,
    structure_names=fs_kcc_ca.structure_names,
)
baseline_ca.split_data(train_size=0.9, random_state=RANDOM_STATE)
baseline_ca.train_model(n_estimators=100, random_state=RANDOM_STATE)
metrics_ca_baseline = baseline_ca.evaluate_model()
_ = baseline_ca.plot_confusion_matrix()

save_ca = os.path.join(KCC_OUTPUT_DIR, "majority_subsampling", "ca")
cmp_ca = FeatureClassification.compare_majority_subsampling_models(
    baseline_ca,
    n_bootstrap=N_BOOTSTRAP,
    n_estimators=100,
    random_state=RANDOM_STATE,
    save_dir=save_ca,
    show_plot=True,
)
print("\nCα comparison complete →", save_ca)
